# Cell-type prediction from methylation `.cz` (`CellTypeClassifier`)

Predict the cell type of a shallow single-cell (e.g. MiSeq snm3C-seq, ~200 reads/cell) `.cz` file using deep-sequencing per-cell-type pseudobulk `.cz` files.

**Method** — a transparent, training-free naive-Bayes / methylation-frequency deconvolution:

1. For each candidate type `t` and cytosine `c`, estimate a *continuous* methylation frequency from the pseudobulk with Beta shrinkage:
   $$\theta_{c,t} = \frac{m_{c,t} + \alpha_0}{n_{c,t} + \alpha_0 + \beta_0}$$
2. Score a query cell by the aggregated per-cytosine Bernoulli log-likelihood, summed in log-space:
   $$\log P(t\mid\text{cell}) \propto \log\pi_t + \lambda_{CG}\!\sum_{c\in CG}[\,s_c\log\theta^{CG}_{c,t} + (1-s_c)\log(1-\theta^{CG}_{c,t})\,] + \lambda_{CH}\!\sum_{c\in CH}[\cdots]$$
3. Softmax over types → calibrated probabilities (supports abstention).

CpG and CpH are modelled as two independent channels; the split is read from the `context` column of a `build_ref` **reference** `.cz` (the cell-type / query files usually store only `mc`/`cov`).

> All inputs (query + pseudobulks) must be aligned to the same reference axis (one row per reference cytosine, in reference order).

In [1]:
import os, struct, tempfile
import numpy as np
import cytozip as czip
from cytozip import CellTypeClassifier

## 1. Build a small self-contained example

Three cell types (`Oligo` / `Astro` / `Micro`) over 200 cytosines (alternating CpG / CpH contexts). The cell-type `.cz` store only `mc` / `cov` (as real per-cell / pseudobulk files do), and a separate **reference** `.cz` (`pos`, `strand`, `context`, from `czip build_ref`) supplies the CpG/CpH context. All files share one axis. In real use these come from `czip allc2cz --reference` (per cell) and `czip merge_cz` / pseudobulk (per cell type).

In [2]:
np.random.seed(0)
d = tempfile.mkdtemp()
S = 200
contexts = [b'CGN' if i % 2 == 0 else b'CAC' for i in range(S)]   # half CpG, half CpH
cg = np.array([i for i in range(S) if i % 2 == 0])
ch = np.array([i for i in range(S) if i % 2 == 1])

# per-type methylation frequency profiles (continuous, discriminative)
prof = {}
f = np.empty(S); f[cg] = 0.90; f[ch] = np.where(np.arange(len(ch)) < len(ch)//2, 0.10, 0.02); prof['Oligo'] = f.copy()
f = np.empty(S); f[cg] = 0.60; f[ch] = np.where(np.arange(len(ch)) < len(ch)//2, 0.02, 0.10); prof['Astro'] = f.copy()
f = np.empty(S); f[cg] = 0.75; f[ch] = 0.05;                                                 prof['Micro'] = f.copy()

def write_cz(path, frac, depth):
    """Write an mc/cov-only .cz (as real per-cell / pseudobulk files are)."""
    cov = np.maximum(np.random.poisson(depth, S), 0).astype(np.uint16)
    mc = np.random.binomial(cov, frac).astype(np.uint16)
    w = czip.Writer(path, formats=['H', 'H'], columns=['mc', 'cov'],
                    chunk_dims=['chrom'], sort_col=False)
    w.write_chunk(b''.join(struct.pack('<HH', int(mc[i]), int(cov[i]))
                           for i in range(S)), ['chr1'])
    w.close()

# a build_ref-style reference .cz carrying pos/strand/context (the CpG/CpH axis)
reference = os.path.join(d, 'reference.cz')
w = czip.Writer(reference, formats=['Q', 'c', '3s'], columns=['pos', 'strand', 'context'],
                chunk_dims=['chrom'], sort_col='pos', delta_cols=['pos'])
w.write_chunk(b''.join(struct.pack('<Qc3s', i + 1, b'+', contexts[i]) for i in range(S)), ['chr1'])
w.close()

# deep pseudobulk cell-type files (mc/cov-only, high coverage)
pseudobulks = {}
for t in prof:
    p = os.path.join(d, f'{t}.cz'); write_cz(p, prof[t], depth=60); pseudobulks[t] = p

# a shallow query cell that is truly Oligo (~3x coverage)
query = os.path.join(d, 'cellQ.cz'); write_cz(query, prof['Oligo'], depth=3)
pseudobulks

{'Oligo': '/tmp/tmpcbb21e54/Oligo.cz',
 'Astro': '/tmp/tmpcbb21e54/Astro.cz',
 'Micro': '/tmp/tmpcbb21e54/Micro.cz'}

## 2. Fit the classifier

`fit` estimates the CpG/CpH frequencies once. The cell-type files are `mc`/`cov`-only, so the CpG/CpH split comes from the `reference` `.cz`. Optional `top_cg` / `top_ch` keep only the most discriminative sites; `cell_counts` enables an abundance / tempered prior at predict time.

In [3]:
clf = CellTypeClassifier(lambda_cg=1.0, lambda_ch=1.0).fit(
    pseudobulks=pseudobulks,
    reference=reference,                                      # build_ref .cz supplying context
    cell_counts={'Oligo': 1000, 'Astro': 500, 'Micro': 50},   # optional (for abundance prior)
    top_cg=None, top_ch=None,                                 # or e.g. 0.1 to keep top 10% sites
)

2026-11-07 21:46:53.299 | INFO     | cytozip.model:fit:1227 - estimating Beta prior on 1 chunk(s) (prior_chroms=['chr1', 'chr2', 'chr3'])


2026-11-07 21:46:53.300 | INFO     | cytozip.model:fit:1278 - CG: Beta prior alpha0=7.951, beta0=2.641 (mean=0.751, kappa=10.59)


2026-11-07 21:46:53.301 | INFO     | cytozip.model:fit:1282 - CH: Beta prior alpha0=2.627, beta0=43.43 (mean=0.057, kappa=46.06)


2026-11-07 21:46:53.301 | INFO     | cytozip.model:fit:1315 - PASS B: scoring cross-type theta range over 3 cell types (100 CG + 100 CH sites)...


2026-11-07 21:46:53.302 | INFO     | cytozip.model:run:1202 - PASS B (theta range): 1/3 cell types done


2026-11-07 21:46:53.302 | INFO     | cytozip.model:run:1202 - PASS B (theta range): 2/3 cell types done


... (5 outputs left out)

2026-11-07 21:46:53.307 | INFO     | cytozip.model:run:1202 - PASS C (gather log-theta): 3/3 cell types done


2026-11-07 21:46:53.307 | INFO     | cytozip.model:fit:1538 - PASS C done; flushing model store to disk...


2026-11-07 21:46:53.308 | INFO     | cytozip.model:fit:1556 - CG: 100 context sites -> 100 discriminative sites across 3 cell types in 1 chunks


2026-11-07 21:46:53.308 | INFO     | cytozip.model:fit:1560 - CH: 100 context sites -> 100 discriminative sites across 3 cell types in 1 chunks


## 3. Predict a single cell

`predict` returns the label, confidence, the full softmax `proba` Series, and the unnormalized `log_posterior`. `prior_alpha` tempers the abundance prior (`0` = uniform, `1` = full abundance).

In [4]:
res = clf.predict(query, prior_alpha=0.0)          # uniform prior
print('label     :', res['label'])
print('confidence: %.4f' % res['confidence'])
res['proba'].round(4)

label     : Oligo
confidence: 1.0000


Oligo    1.0
Astro    0.0
Micro    0.0
dtype: float64

In [5]:
# abstain when the top probability is not confident enough
clf.predict(query, abstain_threshold=0.999)['label']

'Oligo'

## 4. Save the trained model, then predict a new cell after reloading

`save` writes a small **model directory** (`meta.json` + per-channel `log_theta` / `log1m_theta` / `sites` `.npy`, no pickling). `load` memory-maps those arrays, so there is no refitting, **no need to re-read the pseudobulks**, and even a hundreds-of-GB whole-genome model loads in well under a second.


In [6]:
model_dir = os.path.join(d, 'celltype_model')
clf.save(model_dir)
print('model size: %.1f KB' % (sum(os.path.getsize(os.path.join(model_dir, f))
                                      for f in os.listdir(model_dir)) / 1024))
print('files:', sorted(os.listdir(model_dir)))

clf2 = CellTypeClassifier.load(model_dir)
res2 = clf2.predict(query, prior_alpha=1.0)        # abundance prior this time
print('reloaded label:', res2['label'], '(conf %.4f)' % res2['confidence'])


2026-11-07 21:46:58.687 | INFO     | cytozip.model:save:2945 - saved CellTypeClassifier to /tmp/tmpcbb21e54/celltype_model


model size: 6.9 KB
files: ['cg_log1m_theta.npy', 'cg_log_theta.npy', 'cg_sites.npy', 'ch_log1m_theta.npy', 'ch_log_theta.npy', 'ch_sites.npy', 'meta.json']
reloaded label: Oligo (conf 1.0000)


## 5. Batch-predict many cells

`predict_batch` accepts a `{cell_id: path}` dict (or a list of paths) and returns a labels DataFrame plus a cell x cell_type probability matrix.

In [7]:
labels, proba = clf2.predict_batch({'cellQ': query})
display(labels)
proba.round(4)

2026-11-07 21:46:58.709 | DEBUG    | cytozip.model:_score_group:2097 - scoring cells 1-1 of 1 by streaming the model tables


,label,confidence
cellQ,Oligo,1.0


,Oligo,Astro,Micro
cellQ,1.0,0.0,0.0


## 6. Applying it to real data

The cell-type pseudobulk and single-cell `.cz` normally store only `mc`/`cov` and are aligned to the same reference axis (produced by `czip build_ref` + `czip allc2cz --reference`). Pass the `build_ref` reference `.cz` via `reference=` so the classifier can split CpG vs CpH:

```python
from cytozip import CellTypeClassifier, predict_cell_type

# one-shot: fit (or reuse <outdir>/model), predict, write predictions.csv / predict_proba.csv
labels, proba = predict_cell_type(
    query='miseq/cz/',                    # directory, {cell_id: path}, table, or a cat multi-cell .cz
    pseudobulks='pseudobulk/',            # directory of per-type .cz (stem = cell type)
    reference='hg38_with_chrL.allc.cz',   # build_ref .cz supplying the per-row context
    level_adjust=True,                    # per-cell global mC-level offset (MiSeq vs deep atlas)
    n_jobs=-1,                            # stream=None -> table scan chosen automatically
    outdir='~/out/celltype')

# or explicitly: fit once, save, reuse
clf = CellTypeClassifier(lambda_cg=1.0, lambda_ch=1.0).fit(
    pseudobulks='pseudobulk/', reference='hg38_with_chrL.allc.cz',
    top_cg=None, top_ch=None,             # keep all sites: shallow cells need the coverage
    n_jobs=-1, outdir='~/out/celltype/model')
clf.save('~/out/celltype/model')

clf = CellTypeClassifier.load('~/out/celltype/model')     # mmap, ~0.5 s even for 300 GB
labels, proba = clf.predict_batch({cell_id: f'miseq/{cell_id}.cz' for cell_id in cell_ids},
                                  level_adjust=True, abstain_threshold=0.6, n_jobs=-1)
```

**Performance notes**: scoring is sparse-first (cost ∝ the query's covered sites, not the model size) and batches read the model tables with **one sequential scan** instead of per-cell random reads, so a 312 GB all-sites model classifies 1000 MiSeq cells in ~4 min. Keep (nearly) all sites for shallow queries — a 10 k-site MiSeq cell overlaps too few markers of a heavily pruned model.

**Tuning tips**: `level_adjust=True` removes a global mC-level / platform shift between query and reference (on real SALK045 MiSeq cells accuracy went 0.61 → 0.79); start with a uniform prior (`prior_alpha=0`) to check raw discriminative power, then sweep `prior_alpha` / `lambda_ch` on a held-out set while watching **macro-F1** (fair to rare types) and the confusion matrix, not just overall accuracy. For **deep** snm3C queries pass `cov_cap=1`.
